# Parent-Child Chunking — Index Small, Retrieve Big

**The dilemma:**
- Index **full documents** → long docs dilute the relevant parts, search is imprecise
- Index **tiny chunks** → precise matching but you lose surrounding context

**The solution:** Index SMALL chunks (children) for precise matching, but RETURN the
FULL document (parent) so the user gets complete context.

**Analogy:** Like searching a book by its index entries (small, precise) but returning
the full chapter (big, contextual) when you find a match.

In [ ]:
# A long document — searching the full thing is imprecise
parent_doc = (
    "Taiwan Semiconductor Manufacturing Company (TSMC) is the world's largest dedicated "
    "independent semiconductor foundry. Headquartered in Hsinchu, Taiwan, it has a supplier "
    "reliability rating of 94.7% and produces over 50% of the world's outsourced chips. "
    "Key clients include Apple, NVIDIA, and Qualcomm. If TSMC experiences a disruption, "
    "the downstream impact includes: Apple iPhone production halts within 4 weeks, NVIDIA "
    "GPU supply drops by 60%, and automotive chip shortages cascade to 15+ OEMs. "
    "Estimated global revenue impact: $80B annually."
)

print(f"Parent document: {len(parent_doc)} characters")
print(f"\"{parent_doc[:100]}...\"")

In [ ]:
# Split into child chunks
def chunk(text, size=120, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks

children = chunk(parent_doc, size=120, overlap=20)
print(f"Split into {len(children)} child chunks:\n")
for i, c in enumerate(children):
    print(f"  Child {i}: \"{c}\"")
    print(f"           ({len(c)} chars)\n")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

query = "NVIDIA GPU supply disruption"

# Strategy A: Search against full parent document
parent_emb = model.encode(parent_doc)
query_emb = model.encode(query)
parent_sim = cosine_sim(query_emb, parent_emb)

# Strategy B: Search against child chunks
child_embs = model.encode(children)
child_sims = [cosine_sim(query_emb, c) for c in child_embs]
best_child_idx = np.argmax(child_sims)
best_child_sim = child_sims[best_child_idx]

print(f"Query: \"{query}\"\n")
print(f"Strategy A (full doc search):")
print(f"  Similarity: {parent_sim:.4f}")
print(f"  Returns: entire {len(parent_doc)}-char document\n")
print(f"Strategy B (parent-child):")
print(f"  Best child chunk #{best_child_idx}: similarity {best_child_sim:.4f}")
print(f"  Matched: \"{children[best_child_idx]}\"")
print(f"  Returns: entire parent document (full context)\n")
print(f"Child search is {((best_child_sim - parent_sim) / parent_sim * 100):.1f}% more precise!")
print(f"But you still get the full document back — best of both worlds.")

In [ ]:
# Show all child similarities
print(f"All child chunk similarities for \"{query}\":\n")
for i, (c, sim) in enumerate(zip(children, child_sims)):
    bar = '█' * int(sim * 40)
    best = " ← BEST MATCH" if i == best_child_idx else ""
    print(f"  Child {i} ({sim:.4f}): {bar}{best}")
    print(f"    \"{c[:60]}...\"\n")

## Key Takeaways

1. **Small chunks = precise matching** — the relevant 120-char chunk scores higher than the full 500-char doc
2. **Return the parent = full context** — the user sees the complete document, not a fragment
3. **Deduplication matters** — multiple children from the same parent should produce ONE result
4. **Chunk size is a hyperparameter** — smaller chunks = more precise but more to index
5. **Works great with re-ranking** — parent-child retrieval + cross-encoder = very precise results